In [0]:

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from pyspark.sql import functions as F

def load_to_bronze(table_name, source_path, df):
    target_table = get_table_name(table_name, "bronze")
    file_name = source_path.rstrip("/").split("/")[-1]

    # Standardize columns and add metadata
    df = dynamic_column_mapping(df)
    df = add_ingestion_metadata(df, file_name)

    # Create watermark table
    watermark_table = get_table_name("pipeline_watermark", "gold")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

    watermark_schema = StructType([
        StructField("table_name", StringType(), False),
        StructField("file_name", StringType(), False),
        StructField("processed_timestamp", TimestampType(), False)
    ])
    
    spark.createDataFrame([], watermark_schema).write.format("delta").mode("ignore").saveAsTable(watermark_table)

    # Check if already processed
    processed = (
        spark.table(watermark_table)
        .filter(
            (F.col("table_name") == target_table) &
            (F.col("file_name") == file_name)
        )
        .limit(1)
        .count()
    )

    if processed:
        print(f"Skipping already processed file: {file_name}")
        return False

    # Append data to Bronze
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

    # Record successful load
    watermark_df = (
        spark.createDataFrame(
            [(target_table, file_name)],
            ["table_name", "file_name"]
        )
        .withColumn(
            "processed_timestamp",
            F.current_timestamp()
        )
    )

    (
        watermark_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(watermark_table)
    )

    print(f"Loaded {file_name} -> {target_table}")
    return True